# `@wasm` — the full v0.2.5 feature set, run

**PythScribe v0.2.5.** This notebook exercises **every** `@wasm` subsystem shipped in v0.2.5 on real
kernels, with **measured** timings and an oracle for every correctness claim. It is executed
end-to-end; the outputs below are real.

**What `@wasm` is.** You write an ordinary Python numeric kernel, decorate it `@wasm`, and `pyths`
compiles it ahead-of-time to a `.wasm` that runs **in the browser tab**, **in-process on the server**
(wasmtime — sandboxed, GIL-free, bit-for-bit), or as **plain Python** (a load-bearing fallback) — the
*same* function, three engines, identical bits.

**What `@wasm` is NOT (read this first — the honest positioning, per M4).** `@wasm` does **not** beat
NumPy or Numba on vectorised numerics — NumPy is C with SIMD; `@wasm` is a scalar loop. Section 4
proves it with numbers (`@wasm` wins **0** speed columns on the 24 Livermore kernels). The value story
is the **capability sandbox**, **bit-for-bit isomorphism across engines**, a **single deployable
`.wasm`**, **source-compatibility with plain Python**, and a **narrow admission edge** — never "faster
than NumPy", and never "admits far more than Numba".

**Discipline in this notebook.** Every *"the WASM ran"* claim carries a **resolution marker**
(`mode=server` / `path=browser-wasm` / `server_calls` / `python_calls`), not just output parity — a
behaviour-parity pass does **not** prove which path executed. Every *correctness* claim is
`assert == CPython/NumPy`. Every timing is a **measured** `perf_counter` best-of-N — the millisecond
numbers are measurements, never thresholds; the only timing *assertions* are the honest-**direction**
checks (NumPy beats `@wasm` on the vectorised reduction; `@wasm` scores 0 fastest-columns on Livermore),
which are unflippable in our favour.

| Section | Feature |
|---|---|
| 1 | Canonical minimal shape — the demo idiom |
| 2 | Numeric arrays (M2) — 5 dtypes, 1-D and 2-D, buffers not elements + honest timing |
| 3 | The three execution paths — server == browser == CPython, bit-for-bit |
| 4 | The admission edge (honest, M4) — what `@wasm` admits that Numba rejects, and where it loses |
| 5 | Framework adapters (M3) — Gradio + Streamlit in-browser, with the resolution marker |
| 6 | Where it wins / where it loses |

In [1]:
import os, sys, json, time, shutil, subprocess, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np

import pythscribe
from pythscribe import wasm, binding_of
from pythscribe.build import find_pyths, pyths_version
from pythscribe.runtime import ServerKernel, Sandbox, wasmtime_available

# The notebook lives in examples/wasm-use-cases; make its helpers importable whatever the cwd.
_CWD = Path.cwd()
NB_DIR = _CWD if (_CWD / "wasm_full_features.py").is_file() else \
         Path(pythscribe.__file__).resolve().parents[1] / "examples" / "wasm-use-cases"
sys.path.insert(0, str(NB_DIR))
import wasm_full_features as W   # reuses the shipped runtime + M2 shim + M4 harness (no re-marshalling)
import features_lib as F         # measured timing + kernel loading (the M1.5 notebook's library)
import iso_drive                 # the real-tab isomorphic driver (Gradio in a headless Chromium)

PYTHS = find_pyths()
assert wasmtime_available(), "wasmtime is required: pip install -e .[server]"
assert shutil.which("node"), "node is required for the browser-channel (V8 shim) arm"
WORK = NB_DIR / "_v025_work"; WORK.mkdir(exist_ok=True)

# PREFLIGHT: `pyths --version` still prints 0.2.4, but M2 typed-array support is what this notebook
# needs — so probe the CAPABILITY, not the version string. Compile a 1-line Array kernel and confirm
# it WASM-admits; if it does not, the `pyths` on PATH predates M2 (build it: `cargo build --release
# --bin pyths`, or point PYTHSCRIBE_PYTHS at a v0.2.5 build).
_probe = W.compile_wasm("def _p(a: Array[int32], out: Array[int32]) -> None:\n    for i in range(len(a)):\n        out[i] = a[i] + 1\n", WORK, "_preflight")
assert _probe is not None and "_p" in ServerKernel.from_wasm(_probe).exports, (
    "this `pyths` has no M2 typed-array support (Array[dtype] routed to JS) — build a v0.2.5 compiler: "
    "cargo build --release --bin pyths (or set PYTHSCRIBE_PYTHS=<path to a v0.2.5 pyths>)"
)

_node = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout.strip()
print(f"pythscribe {pythscribe.__version__}   pyths {pyths_version(PYTHS)}   ({PYTHS.name})   [M2 array support: OK]")
print(f"python {sys.version.split()[0]}   node {_node}   numpy {np.__version__}")
print(f"wasmtime available: {wasmtime_available()}   |   working dir: {NB_DIR.name}/")

pythscribe 0.2.5a0   pyths 0.2.4   (pyths.exe)   [M2 array support: OK]
python 3.12.7   node v22.15.0   numpy 1.26.4
wasmtime available: True   |   working dir: wasm-use-cases/


## 1 — Canonical minimal shape (the demo idiom)

```python
from pythscribe import wasm

@wasm
def edit_distance(a, b, prev, cur): ...   # ordinary Python; compiled AOT by `pyths`
```

The idiom: call the `@wasm` function, call its **plain-Python twin** (`binding_of(fn).run_python`),
and an **independent oracle**; assert all three agree; time `@wasm` vs Python. `mode=server` is the
marker that the compiled `.wasm` (not the Python body) ran.

In [2]:
# edit_distance ships with a committed .wasm beside kernels.py, so its binding resolves mode=server.
K = F.load_kernels()
b = binding_of(K.edit_distance)
assert b.mode == "server", f"expected the server path, got mode={b.mode} ({b.mode_reason})"

a, s = F.random_text(400, 0), F.random_text(400, 1)
wasm_res = F.edit_distance_call(K.edit_distance, a, s)      # the @wasm (compiled WASM) path
py_res   = F.edit_distance_call(b.run_python, a, s)         # the plain-Python twin, same source
ref      = F.edit_distance_ref(a, s)                        # an INDEPENDENT full-matrix oracle
assert wasm_res == py_res == ref, (wasm_res, py_res, ref)

wasm_s, _ = F.time_best(lambda: F.edit_distance_call(K.edit_distance, a, s), 5)
py_s,   _ = F.time_best(lambda: F.edit_distance_call(b.run_python,   a, s), 2)

# MARKER, asserted: the @wasm calls incremented server_calls (WASM ran); the twin incremented
# python_calls (the Python body ran) — the two counters are distinct, so a WASM run can never be
# confused with a fallback (decorators.py: run_server -> record_server_call; run_python -> record_call).
assert b.server_runs() == 6 and b.calls() == 3, (b.server_runs(), b.calls())  # 1 correctness call + 5 timed; 1 + 2

print(f"edit_distance(400x400):  @wasm = {wasm_res}  ==  python twin = {py_res}  ==  oracle = {ref}")
print(f"time(@wasm)   = {wasm_s*1e3:7.2f} ms")
print(f"time(python)  = {py_s*1e3:7.2f} ms")
print(f"speedup       = {py_s/wasm_s:.1f}x  vs interpreted CPython (a non-vectorisable DP; NOT a NumPy comparison)")
print(f"MARKER:  mode={b.mode}  server_calls={b.server_runs()}  python_calls={b.calls()}  (server_calls>0 ⇒ the WASM ran)")

edit_distance(400x400):  @wasm = 301  ==  python twin = 301  ==  oracle = 301
time(@wasm)   =    4.55 ms
time(python)  =   55.88 ms
speedup       = 12.3x  vs interpreted CPython (a non-vectorisable DP; NOT a NumPy comparison)
MARKER:  mode=server  server_calls=6  python_calls=3  (server_calls>0 ⇒ the WASM ran)


## 2 — Numeric arrays (M2): 5 dtypes, 1-D and 2-D, **buffers not elements**

M2 crosses first-class **typed arrays** — `Array[dtype]` / `Array[dtype, 2]` for
`int32 · int64 · float32 · float64 · uint8` — through the Python buffer protocol
(`np.ndarray` / `memoryview` / `array.array`). The element region is crossed in **one bulk copy**
each way (never element-by-element). The header ABI is
`[dtype:i32][ndim:i32][shape0:i32][shape1:i32]` (16 bytes, elements 8-aligned at `+16`).

Every kernel below uses only ring ops (`+ - *`) and `range(len(a))` indexing, so it WASM-admits on its
dtype (uint8/int32 non-ring ops like `//`,`&`,`>>` and `for x in a` route to JS by the G1/G2
soundness guards — sound-by-refusal, never mis-lowered).

In [3]:
matrix_wasm = W.compile_wasm(W.ARRAY_MATRIX_SRC, WORK, "array_matrix")
assert matrix_wasm is not None, "the array matrix produced no WASM"
KMAT = ServerKernel.from_wasm(matrix_wasm, name="array_matrix")
missing = [f for f in W.ARRAY_MATRIX_FNS if f not in KMAT.exports]
assert not missing, f"routed to JS (array admission did not flip on): {missing}"
print(f"{len(W.ARRAY_MATRIX_FNS)} array kernels compiled and WASM-admitted (all 5 dtypes, 1-D and 2-D):")
for f in W.ARRAY_MATRIX_FNS:
    print("   -", f)
print("\nheader ABI: [dtype:i32][ndim:i32][shape0:i32][shape1:i32]  (16 B; elements 8-aligned at +16)")

# WITNESS the G1/G2 routing (not merely inferred from the matrix admitting): a sub-64-bit non-ring op
# (uint8 //) and array `for x in a` are REFUSED from WASM and routed to JS (sound-by-refusal), while
# their ring / range(len) TWINS stay admitted — the paired positive controls.
guard_src = (
    "def u8avg(a: Array[uint8], b: Array[uint8], out: Array[uint8]) -> None:\n"
    "    for i in range(len(a)):\n"
    "        out[i] = (a[i] + b[i]) // 2\n"
    "def u8add(a: Array[uint8], out: Array[uint8]) -> None:\n"
    "    for i in range(len(a)):\n"
    "        out[i] = a[i] + 1\n"
    "def foriter(a: Array[int64]) -> int:\n"
    "    s = 0\n"
    "    for x in a:\n"
    "        s = s + x\n"
    "    return s\n"
    "def byindex(a: Array[int64]) -> int:\n"
    "    s = 0\n"
    "    for i in range(len(a)):\n"
    "        s = s + a[i]\n"
    "    return s\n"
)
gwasm = W.compile_wasm(guard_src, WORK, "guards")
gexp = set(ServerKernel.from_wasm(gwasm).exports) if gwasm is not None else set()
assert "u8avg" not in gexp and "foriter" not in gexp, f"a G1/G2-guarded op unexpectedly landed on WASM: {gexp}"
assert "u8add" in gexp and "byindex" in gexp, f"a safe ring / range(len) twin was over-refused: {gexp}"
print("routing WITNESSED:  uint8 //  and  `for x in a`  -> JS (refused);   uint8 +  and  range(len)  -> WASM (admitted).")

12 array kernels compiled and WASM-admitted (all 5 dtypes, 1-D and 2-D):
   - add1_i32
   - add1_i64
   - add1_u8
   - scale_f32
   - scale_f64
   - total_f64
   - scale_i32_2d
   - scale_i64_2d
   - scale_f32_2d
   - total_f64_2d
   - threshold_u8_2d
   - downscale_nn

header ABI: [dtype:i32][ndim:i32][shape0:i32][shape1:i32]  (16 B; elements 8-aligned at +16)
routing WITNESSED:  uint8 //  and  `for x in a`  -> JS (refused);   uint8 +  and  range(len)  -> WASM (admitted).


### 1-D per dtype — server path == NumPy (bit-for-bit for int/uint8/float64; ≤tol for float32), with wrap witnesses

In [4]:
def _row(dt, kernel, ptypes, args, out_idx, ref, exact, note):
    KMAT.call(kernel, ptypes, args, read_back=[out_idx], return_type="None")
    got = args[out_idx]
    ok = np.array_equal(got, ref) if exact else np.allclose(got, ref, rtol=1e-6, atol=1e-6)
    # int/uint8 value-equality IS bit-equality; f64 single-op is IEEE-exact (value-equal, i.e.
    # bit-identical for these finite non-(-0.0) results); f32 is compute-in-f64-then-narrow => ≤tol.
    lbl = ("bit-for-bit" if dt in ("int32", "int64", "uint8") else "IEEE-exact (value-equal)") if exact \
          else "≤ tol (f64 compute, narrow on store)"
    return (dt, kernel, lbl, bool(ok), note)

rows = []
# int32 +1, wrap witness at 2**31-1  (arrays are passed as np buffers — NOT element lists)
a = np.array([0, 1, -1, 2**20, 2**31-1], dtype=np.int32)
rows.append(_row("int32", "add1_i32", ["Array[int32]","Array[int32]"], [a, np.zeros_like(a)], 1,
                 a + np.int32(1), True, "wrap: 2^31-1 +1 -> -2^31"))
# int64 +1, wrap witness at 2**63-1
a = np.array([0, 1, -1, 2**40, 2**63-1], dtype=np.int64)
rows.append(_row("int64", "add1_i64", ["Array[int64]","Array[int64]"], [a, np.zeros_like(a)], 1,
                 a + np.int64(1), True, "wrap: 2^63-1 +1 -> -2^63"))
# uint8 +1, wrap witness at 255
a = np.array([0, 1, 128, 254, 255], dtype=np.uint8)
rows.append(_row("uint8", "add1_u8", ["Array[uint8]","Array[uint8]"], [a, np.zeros_like(a)], 1,
                 a + np.uint8(1), True, "wrap: 255 +1 -> 0"))
# float64 * 2.5  (single-op elementwise is IEEE-exact)
a = np.array([0.0, 1.5, -2.25, 3.125, 1e10, -7.0], dtype=np.float64)
rows.append(_row("float64", "scale_f64", ["Array[float64]","Array[float64]","float"], [a, np.zeros_like(a), 2.5], 1,
                 a * 2.5, True, "IEEE-exact single op"))
# float32 * 1.1  (k and inputs NOT exactly representable in f32, so the f64-compute-then-narrow
# path genuinely rounds -> ≤tol vs NumPy's pure-f32, exercising the caveat, not bit-exact)
a = np.random.default_rng(3).standard_normal(64).astype(np.float32)
_f32ref = a * np.float32(1.1)
rows.append(_row("float32", "scale_f32", ["Array[float32]","Array[float32]","float"], [a, np.zeros_like(a), 1.1], 1,
                 _f32ref, False, "rng inputs x1.1 (rounding exercised)"))

print(f"{'dtype':8}{'kernel':12}{'exactness':40}{'==NumPy':8} witness")
for dt, k, ex, ok, note in rows:
    print(f"{dt:8}{k:12}{ex:40}{str(ok):8} {note}")
assert all(ok for *_, ok, _ in rows), "a 1-D dtype row disagreed with NumPy"
print("\nall 5 dtypes agree with NumPy (buffers crossed, not elements).")

dtype   kernel      exactness                               ==NumPy  witness
int32   add1_i32    bit-for-bit                             True     wrap: 2^31-1 +1 -> -2^31
int64   add1_i64    bit-for-bit                             True     wrap: 2^63-1 +1 -> -2^63
uint8   add1_u8     bit-for-bit                             True     wrap: 255 +1 -> 0
float64 scale_f64   IEEE-exact (value-equal)                True     IEEE-exact single op
float32 scale_f32   ≤ tol (f64 compute, narrow on store)    True     rng inputs x1.1 (rounding exercised)

all 5 dtypes agree with NumPy (buffers crossed, not elements).


### 2-D per dtype, a reduction, and a 2-D image op — server path == NumPy

In [5]:
def _seq_sum(flat):   # the kernel sums sequentially in f64; the exact oracle is a sequential Python add
    s = 0.0
    for x in flat: s += x
    return s

def _nn_ref(img, scale, oh, ow):   # independent NumPy nearest-neighbor downscale of [H, W*3] RGB
    H, W3 = img.shape
    hwc = img.reshape(H, W3 // 3, 3)
    return hwc[:oh*scale:scale][:oh, :ow*scale:scale][:, :ow, :].reshape(oh, ow*3).astype(np.uint8)

checks = []
# 2-D int32 scale (NON-square 2x3 — a transpose-offset miscompile would diverge)
a = np.array([[1,2,3],[4,5,6]], dtype=np.int32); out = np.zeros_like(a)
KMAT.call("scale_i32_2d", ["Array[int32, 2]","Array[int32, 2]","int"], [a, out, 10], read_back=[1], return_type="None")
checks.append(("int32   2-D scale x10", np.array_equal(out, a*np.int32(10))))
# 2-D int64 scale
a = np.array([[1,-2],[3,4],[5,-6]], dtype=np.int64); out = np.zeros_like(a)
KMAT.call("scale_i64_2d", ["Array[int64, 2]","Array[int64, 2]","int"], [a, out, 7], read_back=[1], return_type="None")
checks.append(("int64   2-D scale x7", np.array_equal(out, a*np.int64(7))))
# 2-D float32 scale x1.1 (rng inputs -> rounding exercised; ≤tol vs NumPy pure-f32)
a = np.random.default_rng(5).standard_normal((3, 4)).astype(np.float32); out = np.zeros_like(a)
KMAT.call("scale_f32_2d", ["Array[float32, 2]","Array[float32, 2]","float"], [a, out, 1.1], read_back=[1], return_type="None")
checks.append(("float32 2-D scale x1.1 (≤tol)", np.allclose(out, a*np.float32(1.1), rtol=1e-6, atol=1e-6)))
# 2-D uint8 image op: per-pixel threshold
img = np.array([[0,100,200],[128,129,255]], dtype=np.uint8); out = np.zeros_like(img)
KMAT.call("threshold_u8_2d", ["Array[uint8, 2]","Array[uint8, 2]"], [img, out], read_back=[1], return_type="None")
checks.append(("uint8   2-D threshold>128", np.array_equal(out, np.where(img > np.uint8(128), np.uint8(255), np.uint8(0)))))
# reductions (scalar return over an array): 1-D and 2-D, exact vs sequential oracle
a1 = np.array([0.5, 1.5, 2.0, -1.0, 4.25, 0.0], dtype=np.float64)
r1 = KMAT.call("total_f64", ["Array[float64]"], [a1], read_back=[], return_type="float").value
checks.append(("float64 1-D reduction (sum)", r1 == _seq_sum(a1.tolist())))
a2 = np.array([[0.5,1.5],[2.0,-1.0],[4.25,0.0]], dtype=np.float64)
r2 = KMAT.call("total_f64_2d", ["Array[float64, 2]"], [a2], read_back=[], return_type="float").value
checks.append(("float64 2-D reduction (sum)", r2 == _seq_sum(a2.reshape(-1).tolist())))
# the flagship 2-D image op: uint8 nearest-neighbor downscale (a real window op)
rng = np.random.default_rng(7); img = rng.integers(0, 256, size=(8, 6*3), dtype=np.uint8)
scale, oh, ow = 2, 4, 3; out = np.zeros((oh, ow*3), dtype=np.uint8)
n = KMAT.call("downscale_nn", ["Array[uint8, 2]","int","int","int","Array[uint8, 2]"], [img, scale, oh, ow, out],
              read_back=[4], return_type="int").value
checks.append(("uint8   2-D NN downscale (flagship)", np.array_equal(out, _nn_ref(img, scale, oh, ow)) and n == oh*ow))

for name, ok in checks:
    print(f"  {name:36} == NumPy: {ok}")
assert all(ok for _, ok in checks), "a 2-D / reduction / image-op row disagreed with NumPy"
print("\nall 2-D dtypes, both reductions, and the NN image op agree with NumPy.")

  int32   2-D scale x10                == NumPy: True
  int64   2-D scale x7                 == NumPy: True
  float32 2-D scale x1.1 (≤tol)        == NumPy: True
  uint8   2-D threshold>128            == NumPy: True
  float64 1-D reduction (sum)          == NumPy: True
  float64 2-D reduction (sum)          == NumPy: True
  uint8   2-D NN downscale (flagship)  == NumPy: True

all 2-D dtypes, both reductions, and the NN image op agree with NumPy.


### Honest timing — `@wasm` vs plain Python vs **NumPy** (NumPy wins on vectorised numerics; we say so)

In [6]:
# A vectorisable reduction: @wasm crosses n floats + runs a scalar loop; NumPy is C with SIMD.
n = 1_000_000
xs = np.random.default_rng(0).standard_normal(n)          # float64 buffer
py_list = xs.tolist()
wasm_s, wv = F.time_best(lambda: KMAT.call("total_f64", ["Array[float64]"], [xs], read_back=[], return_type="float").value, 3)
py_s,   pv = F.time_best(lambda: _seq_sum(py_list), 1)
np_s,   nv = F.time_best(lambda: float(np.sum(xs)), 3)
# values agree to summation-order ulps (kernel/Python are sequential; NumPy is pairwise)
assert wv == pv and abs(nv - wv) <= 1e-6 * abs(nv)
print(f"sum of {n:,} float64:")
print(f"  NumPy   = {np_s*1e3:8.3f} ms   (vectorised C — the fastest)")
print(f"  @wasm   = {wasm_s*1e3:8.3f} ms   (crosses {n:,} floats + a scalar WASM loop)")
print(f"  Python  = {py_s*1e3:8.3f} ms   (interpreted loop)")
print(f"  -> NumPy is {wasm_s/np_s:.0f}x faster than @wasm, and @wasm is {py_s/wasm_s:.1f}x faster than the Python loop.")
assert np_s < wasm_s, "NumPy must win the vectorised reduction (the honest expectation)"
print("HONEST: @wasm beats the interpreted loop but LOSES to NumPy on vectorised numerics — exactly as expected (see section 4).")

sum of 1,000,000 float64:
  NumPy   =    0.626 ms   (vectorised C — the fastest)
  @wasm   =   10.915 ms   (crosses 1,000,000 floats + a scalar WASM loop)
  Python  =   30.087 ms   (interpreted loop)
  -> NumPy is 17x faster than @wasm, and @wasm is 2.8x faster than the Python loop.
HONEST: @wasm beats the interpreted loop but LOSES to NumPy on vectorised numerics — exactly as expected (see section 4).


## 3 — The three execution paths: server == browser == CPython, **bit-for-bit**

The *same* compiled `.wasm`, run three ways:

1. **server** — in-process under wasmtime (`array_buffer.py` marshalling);
2. **browser channel** — the exact `list_buffer.mjs` shim the Gradio/Streamlit components ship,
   driven under Node/V8 (this is the browser *channel* — V8 + the component's own shim — **not** a
   literal tab; the real tab is the next cell);
3. **CPython / NumPy** — the independent oracle.

Asserted **bit-for-bit** for int32/int64/uint8, **IEEE-exact** for single-op float64, and **≤tol** for
float32 (both engines narrow f64→f32 identically). The kernel's **scalar return** crosses on both paths
too (checked for the flagship).

In [7]:
def _iso(fn, ptypes, args_server, args_shim, out_idx, dtype, ndim, ref, return_type, exact=True):
    rs = KMAT.call(fn, ptypes, args_server, read_back=[out_idx], return_type=return_type)
    r = W.shim_array_call(matrix_wasm, fn, ptypes, args_shim, [out_idx], return_type)
    assert r.get("ok"), r
    out_s, out_b = args_server[out_idx], W.shim_readback_np(r["readback"][0], dtype, ndim)
    # server and browser run the SAME .wasm, so their outputs are ALWAYS bit-identical (f32 included);
    # only the comparison to the NumPy ORACLE is ≤tol for f32 (both engines narrow f64->f32; NumPy is
    # pure-f32). So: server == browser is asserted bit-exact; server == ref is exact (int/uint8/f64) or ≤tol (f32).
    eq_ref = np.array_equal if exact else (lambda x, y: np.allclose(x, y, rtol=1e-6, atol=1e-6))
    same = bool(np.array_equal(out_s, out_b) and eq_ref(out_s, ref))
    if return_type == "int":                         # the scalar return crosses on BOTH paths too
        same = same and (int(r["ret"]) == rs.value)
    return out_s, out_b, same

arms = []
# int32 1-D (Int32Array), wrap witness at 2^31-1
a = np.array([0, 1, -1, 2**20, 2**31-1], dtype=np.int32)
_, _, ok = _iso("add1_i32", ["Array[int32]","Array[int32]"], [a, np.zeros_like(a)], [a, np.zeros_like(a)],
                1, "int32", 1, a + np.int32(1), "None"); arms.append(("int32   1-D add1 (2^31-1 wrap)", ok))
# int64 1-D (BigInt64Array), wrap witness at 2^63-1
a = np.array([0, 1, -1, 2**40, 2**63-1], dtype=np.int64)
_, _, ok = _iso("add1_i64", ["Array[int64]","Array[int64]"], [a, np.zeros_like(a)], [a, np.zeros_like(a)],
                1, "int64", 1, a + np.int64(1), "None"); arms.append(("int64   1-D add1 (2^63-1 wrap)", ok))
# float64 1-D (bit-for-bit single op)
a = np.array([0.0, 1.5, -2.25, 3.125, 1e10], dtype=np.float64)
_, _, ok = _iso("scale_f64", ["Array[float64]","Array[float64]","float"], [a, np.zeros_like(a), 2.5],
                [a, np.zeros_like(a), 2.5], 1, "float64", 1, a * 2.5, "None"); arms.append(("float64 1-D scale x2.5 (IEEE-exact)", ok))
# float32 1-D (≤tol — both engines compute-in-f64 then narrow, so they AGREE within tol)
a = np.random.default_rng(9).standard_normal(48).astype(np.float32)
_, _, ok = _iso("scale_f32", ["Array[float32]","Array[float32]","float"], [a, np.zeros_like(a), 1.1],
                [a, np.zeros_like(a), 1.1], 1, "float32", 1, a * np.float32(1.1), "None", exact=False); arms.append(("float32 1-D scale x1.1 (≤tol)", ok))
# uint8 2-D nearest-neighbor downscale — the flagship, bit-for-bit across all three engines (+ scalar return)
rng = np.random.default_rng(4242); img = rng.integers(0, 256, size=(9, 7*3), dtype=np.uint8)
scale, oh, ow = 3, 3, 2
ref = _nn_ref(img, scale, oh, ow)
ptypes = ["Array[uint8, 2]","int","int","int","Array[uint8, 2]"]
outs, outb, ok = _iso("downscale_nn", ptypes, [img, scale, oh, ow, np.zeros((oh, ow*3), np.uint8)],
                      [img, scale, oh, ow, np.zeros((oh, ow*3), np.uint8)], 4, "uint8", 2, ref, "int")
arms.append(("uint8   2-D NN downscale (flagship, +scalar return)", ok))

print("server == browser(V8 shim) == CPython/NumPy across the dtype matrix:")
for name, ok in arms:
    print(f"  {name:44} {ok}")
assert all(ok for _, ok in arms), "an isomorphic (server==browser==CPython) check failed"
print("\nflagship downscale — server bytes[:6] =", outs.reshape(-1)[:6].tolist(),
      "| browser bytes[:6] =", outb.reshape(-1)[:6].tolist(), "(identical)")

server == browser(V8 shim) == CPython/NumPy across the dtype matrix:
  int32   1-D add1 (2^31-1 wrap)               True
  int64   1-D add1 (2^63-1 wrap)               True
  float64 1-D scale x2.5 (IEEE-exact)          True
  float32 1-D scale x1.1 (≤tol)                True
  uint8   2-D NN downscale (flagship, +scalar return) True

flagship downscale — server bytes[:6] = [166, 8, 130, 67, 25, 219] | browser bytes[:6] = [166, 8, 130, 67, 25, 219] (identical)


### The **real tab** — a Gradio app in a headless Chromium, with the resolution marker

In [8]:
# One @wasm function (dtw_distance) run IN A REAL BROWSER TAB and in-process on the server; the tab's
# IEEE-754 bits are compared to the server's and to CPython's. The MARKER (not just parity) proves the
# WASM ran in the tab: path=browser-wasm, the tab fetched the .wasm, and the server did NOTHING for
# THIS CLICK. Note `mode=server` (the binding's resolved mode) sits beside `python_calls=server_calls=0`
# with no contradiction: those two are the per-dispatch DELTAS the adapter records (app.py). Each click
# computes the CPython/server reference via the binding's Python body BEFORE calling dispatch() — the
# server_calls delta is measured across dispatch only, so the tab did the WASM work and the click drove
# zero server-WASM runs. Runs the driver in a subprocess (Playwright's sync API + the notebook's asyncio loop).
xa = [1.0, 2.0, 3.0, 4.0, 5.0, 4.0, 3.0]
xb = [1.0, 2.0, 2.0, 3.0, 5.0, 5.0, 3.0]
iso = iso_drive.measure_isomorphic(xa, xb)
print(json.dumps({k: iso[k] for k in ("path","browser_bits","server_bits","cpython_bits","identical",
                                      "mode","python_calls","server_calls","wasm_fetched")}, indent=2))
assert iso["path"] == "browser-wasm", iso["path"]
assert iso["browser_bits"] == iso["server_bits"] == iso["cpython_bits"] and iso["identical"]
assert iso["python_calls"] == 0 and iso["server_calls"] == 0, "the server must have done nothing"
assert iso["wasm_fetched"] and not iso["console_errors"]
print("\nMARKER asserted: the .wasm ran IN THE TAB (browser-wasm, wasm_fetched), the server did nothing,")
print("and browser == server == CPython bits are identical — a true isomorphic run, not a parity coincidence.")

{
  "path": "browser-wasm",
  "browser_bits": "0000000000000040",
  "server_bits": "0000000000000040",
  "cpython_bits": "0000000000000040",
  "identical": true,
  "mode": "server",
  "python_calls": 0,
  "server_calls": 0,
  "wasm_fetched": true
}

MARKER asserted: the .wasm ran IN THE TAB (browser-wasm, wasm_fetched), the server did nothing,
and browser == server == CPython bits are identical — a true isomorphic run, not a parity coincidence.


## 4 — The admission edge (honest, per M4)

`@wasm`'s admission edge over Numba is **narrow and real**: a `@wasm` kernel that `raise`s a built-in
exception and catches it with `except <SpecificClass>:` is admitted to the WASM fast path — a shape
Numba's nopython frontend **rejects**. Numba *does* accept `raise`, bare `except:`, `except Exception:`,
`typed.Dict`, and `@jitclass`, so the edge is **only** the specific-class `except`, **not** the broad
"Numba rejects classes/dicts/generators" overclaim (M4 corrected that). And **dicts/classes stay on the
JS path** — they never reach wasmtime.

Then the honest counterweight: on vectorised numerics `@wasm` **loses** — the M4 Livermore harness
below scores `@wasm` at **0** fastest-columns.

In [9]:
# (a) the genuine admission win: try/except <SpecificClass> over an explicit raise
gw = W.compile_wasm(W.GUARDED_SUM_SRC, WORK, "guarded_sum")
assert gw is not None and "guarded_sum" in ServerKernel.from_wasm(gw).exports, "guarded_sum did NOT land on WASM"
gk = ServerKernel.from_wasm(gw, name="guarded_sum", sandbox=Sandbox(fuel=50_000_000))
r = gk.call("guarded_sum", ["int"], [1000], return_type="int")
assert r.value == W.guarded_sum_ref(1000) and r.fuel_used > 0
print(f"try/except ValueError kernel LANDS on WASM: guarded_sum(1000) = {r.value} == CPython {W.guarded_sum_ref(1000)}  (fuel_used={r.fuel_used})")

# (a') anti-strawman: Numba genuinely REJECTS the very same kernel
import numba
from numba import njit
try:
    njit(W.guarded_sum_ref)(100); rejected = False; why = "(accepted?!)"
except numba.core.errors.NumbaError as e:
    rejected = True; why = type(e).__name__
assert rejected, "Numba unexpectedly accepted the kernel — the admission win would be a strawman"
print(f"Numba @njit REJECTS the same kernel: {why}")

# (a'') WITNESS the NARROW-ness (not just assert it): Numba ACCEPTS the broad/bare `except` twins of the
# very same kernel — so the genuine edge is ONLY the specific-class catch, nothing broader.
def _bare_except_ref(n):
    s = 0; i = 0
    while i < n:
        try:
            if i % 7 == 0:
                raise ValueError
            s = s + i
        except:            # noqa: E722  (bare except — Numba accepts this)
            s = s - 1
        i = i + 1
    return s
def _broad_except_ref(n):
    s = 0; i = 0
    while i < n:
        try:
            if i % 7 == 0:
                raise ValueError
            s = s + i
        except Exception:  # Numba accepts `except Exception`
            s = s - 1
        i = i + 1
    return s
twins = []
for label, fn in (("bare  except:", _bare_except_ref), ("except Exception:", _broad_except_ref)):
    try:
        v = njit(fn)(200); twins.append((label, True, v == fn(200)))
    except numba.core.errors.NumbaError as e:
        twins.append((label, False, type(e).__name__))
for label, accepted, detail in twins:
    print(f"Numba ACCEPTS the `{label}` twin: {accepted}  (== CPython: {detail})")
assert all(acc and det is True for _, acc, det in twins), "Numba should ACCEPT the broad/bare twins — the edge is only the specific class"

# (b) dicts/classes are NOT WASM-eligible — a dict kernel stays on the JS path
dw = W.compile_wasm(W.DICT_MODE_SRC, WORK, "dict_mode")
dict_on_wasm = dw is not None and "dict_mode" in ServerKernel.from_wasm(dw).exports
assert not dict_on_wasm, "a dict kernel unexpectedly landed on WASM"
print("dict kernel STAYS on the JS path (dicts/classes are not WASM-eligible).")
print("\nHONEST: the admission edge is NARROW — ONLY a specific-class `except` over an explicit `raise`.")
print("Numba accepts raise / bare-except / except Exception / typed.Dict / jitclass (two witnessed above).")

try/except ValueError kernel LANDS on WASM: guarded_sum(1000) = 428286 == CPython 428286  (fuel_used=75726)


Numba @njit REJECTS the same kernel: TypingError


Numba ACCEPTS the `bare  except:` twin: True  (== CPython: True)
Numba ACCEPTS the `except Exception:` twin: True  (== CPython: True)
dict kernel STAYS on the JS path (dicts/classes are not WASM-eligible).

HONEST: the admission edge is NARROW — ONLY a specific-class `except` over an explicit `raise`.
Numba accepts raise / bare-except / except Exception / typed.Dict / jitclass (two witnessed above).


### The counterweight — the M4 Livermore server-row harness (all 24 kernels): `@wasm` wins **0** speed columns

In [10]:
livermore = W.run_livermore()   # the SHIPPED benchmarks/livermore_server_row/run.py, all 24 kernels
print(livermore)
assert "@wasm 0" in livermore, "expected @wasm to win 0 fastest-columns"
assert "ADMISSION wins (@wasm ran a kernel Numba REJECTED): 0" in livermore, "expected 0 Livermore admission wins"
print("CONFIRMED: NumPy/Numba own every speed column; @wasm wins 0. `@wasm` is NOT for vectorised numerics.")


Livermore SERVER-row timings (ms, best of N reps; lower is faster)
kernel                   CPython              NumPy                  Numba     @wasm   ratios (x CPython)
----------------------------------------------------------------------------------------------------------------------
k01_hydro                  0.313              0.020                  0.008     0.151   np  15.3x  nb  40.6x  ws   2.1x
k02_iccg                   0.182  n/a: loop-carried                  0.004     0.187   nb  43.3x  ws   1.0x
k03_inner_product          0.701              0.017                  0.012     0.218   np  40.5x  nb  56.5x  ws   3.2x
k04_banded_linear          6.602  n/a: loop-carried                  0.078     0.413   nb  84.3x  ws  16.0x
k05_tridiag                0.758  n/a: loop-carried                  0.021     0.197   nb  35.9x  ws   3.8x
k06_recurrence             1.074  n/a: loop-carried                  0.020     0.201   nb  52.9x  ws   5.3x
k07_state_eq               2.133     

## 5 — Framework adapters (M3): Gradio + Streamlit, in-browser, with the resolution marker

The adapters run a `@wasm` kernel's compiled `.wasm` **in the user's tab**. Per the dual-track-masking
rule, a behaviour-parity pass is **not** proof the WASM ran. What `dispatch()` gives us, server-side and
**deterministically**, is the *resolution decision*: it **selects the browser path** — it ships the
`.wasm` to the client (Gradio as a fetched `bundle`, Streamlit as embedded `wasm_b64` bytes) and produces
**no** Python `result` (`result=None` ⇒ the browser will compute; `python_calls` stays 0). Note the
payload has **no `path` field** — that key is written by the *browser* into `result` once it runs; below
we print only the fields `dispatch()` actually returns and label them honestly.

The **end-to-end** proof that the `.wasm` truly executed in a tab (`path=browser-wasm`, `wasm_fetched`,
`python_calls=0`, `server_calls=0`, identical bits) was already asserted against a **real Gradio tab** in
section 3. This notebook does **not** drive a real Streamlit tab — the Streamlit evidence here is the
adapter's *resolution decision* only (a real Streamlit-in-tab run is the `examples/streamlit-wasm` app +
its test).

In [11]:
import importlib.util as _u
import pythscribe.gradio as GR
import pythscribe.streamlit as ST

def _load(path, name):
    sp = _u.spec_from_file_location(name, path); m = _u.module_from_spec(sp); sp.loader.exec_module(m); return m

# rms_gain — a float-return scalar kernel with a committed .wasm, in both example apps.
gm = _load(NB_DIR.parent / "gradio-wasm" / "kernels.py", "gr_rms")
sm = _load(NB_DIR.parent / "streamlit-wasm" / "kernels.py", "st_rms")
gb, sb = binding_of(gm.rms_gain), binding_of(sm.rms_gain)
xs, target = [0.1, 0.5, -0.3, 0.8, -0.2], 0.25

gp = GR.dispatch(gm.rms_gain, xs, target)     # Gradio adapter: build the component value for one call
sp = ST.dispatch(sm.rms_gain, xs, target)     # Streamlit adapter: same, wasm embedded as base64

# MARKER (the resolution DECISION, from the fields dispatch actually returns): the browser path is
# selected (the .wasm is shipped to the client) and Python did NOT run (result is None, calls()==0).
g_selects_browser = gp["artifact_status"] == "resolved" and gp["bundle"] is not None and gp["result"] is None
s_selects_browser = sp["artifact_status"] == "resolved" and sp["wasm_b64"] is not None and sp["result"] is None
assert g_selects_browser and gb.calls() == 0, gp
assert s_selects_browser and sb.calls() == 0, sp
print(f"Gradio    dispatch -> browser path SELECTED  (bundle shipped={gp['bundle'] is not None}, result=None ⇒ browser computes)  python_calls={gb.calls()}")
print(f"Streamlit dispatch -> browser path SELECTED  (wasm_b64 shipped={sp['wasm_b64'] is not None}, {sp['wasm_bytes']} B; result=None ⇒ browser computes)  python_calls={sb.calls()}")
print("   (the payload carries no `path` key — that is written by the browser into `result`; see section 3 for the real-tab path marker)")
print()
print("End-to-end proof the WASM ran in a tab (section 3, a real Gradio tab):")
print(f"  path={iso['path']}  wasm_fetched={iso['wasm_fetched']}  python_calls={iso['python_calls']}  server_calls={iso['server_calls']}  browser==server==CPython={iso['identical']}")

Gradio    dispatch -> browser path SELECTED  (bundle shipped=True, result=None ⇒ browser computes)  python_calls=0
Streamlit dispatch -> browser path SELECTED  (wasm_b64 shipped=True, 475 B; result=None ⇒ browser computes)  python_calls=0
   (the payload carries no `path` key — that is written by the browser into `result`; see section 3 for the real-tab path marker)

End-to-end proof the WASM ran in a tab (section 3, a real Gradio tab):
  path=browser-wasm  wasm_fetched=True  python_calls=0  server_calls=0  browser==server==CPython=True


## 6 — Where it wins / where it loses

In [12]:
wins = [
    "Capability sandbox + fuel metering — run untrusted / LLM-generated code with no ambient I/O (server).",
    "Bit-for-bit isomorphism — the SAME .wasm gives identical bits in the tab, in-process, and in CPython.",
    "Single deployable artifact — one .wasm, no toolchain or interpreter shipped at run time.",
    "Source-compatible — the kernel is ordinary Python; the plain-Python fallback is load-bearing.",
    "GIL-free fan-out + fixed float/int semantics on the server path.",
    "A NARROW admission edge — a specific-class `except` over an explicit `raise` that Numba rejects.",
]
loses = [
    "Vectorised numerics — NumPy (C+SIMD) and Numba beat @wasm; on Livermore, @wasm wins 0 speed columns.",
    "Per-element boundary crossing — marshalling dominates a trivial loop; batch the call.",
    "dicts / classes / generators / array `for x in a` / sub-64-bit non-ring ops — routed to JS, not WASM.",
]
print("WHERE @wasm WINS:")
for w in wins: print("  +", w)
print("\nWHERE @wasm LOSES (say so):")
for l in loses: print("  -", l)

print("\n--- consolidated evidence from this run ---")
print(f"  isomorphic real tab:  browser==server==CPython bits = {iso['browser_bits']}  (identical={iso['identical']}, path={iso['path']})")
print(f"  Livermore harness:    " + [ln for ln in livermore.splitlines() if 'tally' in ln][0].strip())
print(f"  array matrix:         {len(W.ARRAY_MATRIX_FNS)} kernels, 5 dtypes x (1-D, 2-D), all == NumPy")
print("\nThe pitch is trust and deployability (sandbox / isomorphic / single-artifact / source-compat),")
print("plus a narrow admission edge — NOT 'faster than NumPy' and NOT 'admits far more than Numba'.")

WHERE @wasm WINS:
  + Capability sandbox + fuel metering — run untrusted / LLM-generated code with no ambient I/O (server).
  + Bit-for-bit isomorphism — the SAME .wasm gives identical bits in the tab, in-process, and in CPython.
  + Single deployable artifact — one .wasm, no toolchain or interpreter shipped at run time.
  + Source-compatible — the kernel is ordinary Python; the plain-Python fallback is load-bearing.
  + GIL-free fan-out + fixed float/int semantics on the server path.
  + A NARROW admission edge — a specific-class `except` over an explicit `raise` that Numba rejects.

WHERE @wasm LOSES (say so):
  - Vectorised numerics — NumPy (C+SIMD) and Numba beat @wasm; on Livermore, @wasm wins 0 speed columns.
  - Per-element boundary crossing — marshalling dominates a trivial loop; batch the call.
  - dicts / classes / generators / array `for x in a` / sub-64-bit non-ring ops — routed to JS, not WASM.

--- consolidated evidence from this run ---
  isomorphic real tab:  browser==s